## Installation des packages

In [34]:
import subprocess
import sys

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "plotly",
]

def install (pack):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pack, "--quiet"])

for pack in packages:
    install(pack)

print("Package installes !")

Package installes !


## Verifications

In [35]:
import os

fichiers_necessaires = [
    "datasets-project/datasets/orders.csv",
    "datasets-project/datasets/products.csv",
    "datasets-project/datasets/aisles.csv",
    "datasets-project/datasets/departments.csv",
    "datasets-project/datasets/order_products.csv",
]

for fichier in fichiers_necessaires:
    if os.path.exists(fichier):
        print(f"{fichier} trouve")
    else:
        print(f"{fichier} MANQUANT")

datasets-project/datasets/orders.csv trouve
datasets-project/datasets/products.csv trouve
datasets-project/datasets/aisles.csv trouve
datasets-project/datasets/departments.csv trouve
datasets-project/datasets/order_products.csv trouve


## Import des packages

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Imports OK !")

Imports OK !


## Preprocessing des donnees

**Chargement de donnees**

In [37]:

DATA_PATH = "datasets-project/datasets"

orders = pd.read_csv(os.path.join(DATA_PATH, "orders.csv"))
products = pd.read_csv(os.path.join(DATA_PATH, "products.csv"))
aisles = pd.read_csv(os.path.join(DATA_PATH, "aisles.csv"))
departments = pd.read_csv(os.path.join(DATA_PATH, "departments.csv"))
order_products = pd.read_csv(os.path.join(DATA_PATH, "order_products.csv"))

print("orders :", orders.shape)
print("products :", products.shape)
print("aisles :", aisles.shape)
print("departments :", departments.shape)
print("order_products :", order_products.shape)

orders : (1444444, 7)
products : (49688, 4)
aisles : (134, 2)
departments : (21, 2)
order_products : (13692886, 4)


**Exploration des fichiers**

In [38]:
print("ORDERS")
display(orders.head(3))
print(orders.dtypes)

print("\nPRODUCTS")
display(products.head(3))

print("\nAISLES")
display(aisles.head(3))

print("\nDEPARTEMENTS")
display(departments.head(3))

print("\nORDER_PRODUCTS")
display(order_products.head(3))

ORDERS


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1.0,1.0,2.0,8.0,NaN,NaN
1,2398795,1.0,2.0,3.0,7.0,15.0,NaN
2,473747,1.0,3.0,3.0,12.0,21.0,NaN


order_id                    int64
user_id                   float64
eval_set                  float64
order_number              float64
order_dow                 float64
order_hour_of_day         float64
days_since_prior_order    float64
dtype: object

PRODUCTS


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7



AISLES


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars



DEPARTEMENTS


,department_id,department
0,1,frozen
1,2,other
2,3,bakery



ORDER_PRODUCTS


,order_id,product_id,add_to_cart_order,reordered
0,2,33120.0,1.0,1.0
1,2,28985.0,2.0,1.0
2,2,9327.0,3.0,0.0


**Correction des colonnes de orders**

In [39]:
orders = pd.read_csv(os.path.join(DATA_PATH, "orders.csv"))

orders = orders.rename(columns={
    "order_number"           : "order_dow",
    "order_dow"              : "order_hour_of_day", 
    "order_hour_of_day"      : "days_since_prior_order",
    "days_since_prior_order" : "order_number"
})

orders = orders.drop(columns=["order_number"])

print(orders.columns.tolist())
print(orders.head(3))
print("\nNaN par colonne :")
print(orders.isnull().sum())

['order_id', 'user_id', 'eval_set', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
   order_id  user_id  eval_set  order_dow  order_hour_of_day  \
0   2539329      1.0       1.0        2.0                8.0   
1   2398795      1.0       2.0        3.0                7.0   
2    473747      1.0       3.0        3.0               12.0   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  

NaN par colonne :
order_id                      0
user_id                      31
eval_set                     23
order_dow                    22
order_hour_of_day            23
days_since_prior_order    86909
dtype: int64


**Fusionnage**

In [40]:
df = order_products.merge(orders, on="order_id", how="left")
df = df.merge(products, on="product_id", how="left")
df = df.merge(aisles, on="aisle_id", how="left")
df = df.merge(departments, on="department_id", how="left")

print("Shape après fusion :", df.shape)

Shape après fusion : (13692886, 14)


## Nettoyage

In [41]:
# days_since_prior_order : NaN = première commande --> remplace par 0
df["days_since_prior_order"] = df["days_since_prior_order"].fillna(0)

#Supprimer les lignes avec NaN
df = df.dropna(subset=[
    "user_id",
    "order_dow",
    "order_hour_of_day",
    "product_name",
    "aisle",
    "department",
    "reordered",
    "add_to_cart_order",
    "aisle_id", 
    "department_id"
])

#Correction = float devient int
df["user_id"] = df["user_id"].astype(int)
df["order_dow"] = df["order_dow"].astype(int)
df["order_hour_of_day"] = df["order_hour_of_day"].astype(int)
df["days_since_prior_order"] = df["days_since_prior_order"].astype(int)
df["product_id"] = df["product_id"].astype(int)
df["reordered"] = df["reordered"].astype(int)
df["add_to_cart_order"] = df["add_to_cart_order"].astype(int)
df["aisle_id"] = df["aisle_id"].astype(int)
df["department_id"] = df["department_id"].astype(int)

df = df.drop(columns=["eval_set"])

#Suppression des doublons
df = df.drop_duplicates()

**Verifications**

In [42]:
print("Shape final :", df.shape)
print("NaN restants :", df.isnull().sum().sum())
print("Doublons restants :", df.duplicated().sum())
print("\nTypes finaux :")
print(df.dtypes)
display(df.head(5))

Shape final : (13690784, 13)
NaN restants : 0
Doublons restants : 0

Types finaux :
order_id                  int64
product_id                int64
add_to_cart_order         int64
reordered                 int64
user_id                   int64
order_dow                 int64
order_hour_of_day         int64
days_since_prior_order    int64
product_name                str
aisle_id                  int64
department_id             int64
aisle                       str
department                  str
dtype: object


,order_id,product_id,add_to_cart_order,reordered,user_id,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,5,9,8,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,5,9,8,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,5,9,8,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,5,9,8,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,5,9,8,Natural Sweetener,17,13,baking ingredients,pantry
